# Random Forest Modeling — Sprint 6

**Overview**
This notebook implements a Random Forest modeling workflow using Gold V2 data.

The notebook includes feature generation, model training, and evaluation for station-level hourly demand signals.

**Model Training**

This section builds temporal features and evaluates Random Forest models using a progressive monthly approach.

**Feature Engineering**

Features are generated within this notebook, including:
- Lag variables (lag1, lag24, lag168)
- Time-based features (hour, day of week)

These features capture short-term trends and periodic patterns in bike demand.

In [0]:
# ============================================================
# Train DEP + ARR from GOLD_V2 - RANDOM FOREST VERSION
# Step A: Build temporal features (lags + rolling) and store once
# Step B: Progressive monthly train/eval using Hash Compact encoding
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

import pandas as pd
import numpy as np
import zlib

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ------------------------------------------------------------
# 0) PATHS
# ------------------------------------------------------------
GOLD_V2_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/gold/gold_v2_spatiotemporal_events"
FEATURES_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/features/goldv2_features_v1"

EVAL_DIR_DEP = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/model_eval/goldv2_dep_rf_v1"
EVAL_DIR_ARR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/model_eval/goldv2_arr_rf_v1"

REBUILD_FEATURES = True 
CACHE_FEATURES   = False 

# ------------------------------------------------------------
# 1) CONFIG
# ------------------------------------------------------------
DOWNTOWN_LAT_MIN, DOWNTOWN_LAT_MAX = 43.63, 43.67
DOWNTOWN_LON_MIN, DOWNTOWN_LON_MAX = -79.41, -79.37

TRAIN_LOOKBACK_DAYS = 90
HASH_BUCKETS = 512
BUCKET_COL = "station_bucket"

# Grid for Random Forest (n_estimators)
N_ESTIMATORS_GRID = [100, 200, 300] # RF is slower 


base_rf_params = dict(
    n_estimators=200,         
    max_depth=12,              
    min_samples_leaf=4,
    max_features="sqrt",       
    bootstrap=True,
    n_jobs=-1,                 
    random_state=42
)

START_Y, START_M = 2022, 10
END_Y, END_M     = 2024,  9

# ------------------------------------------------------------
# 2) HELPERS
# ------------------------------------------------------------
def path_exists(path: str) -> bool:
    try:
        _ = dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def station_to_bucket(station_id: str, n_buckets: int) -> int:
    if station_id is None:
        return 0
    b = str(station_id).encode("utf-8")
    return zlib.crc32(b) % n_buckets

def prev_months_in_lookback(test_start: pd.Timestamp, lookback_days: int):
    start = (test_start - pd.Timedelta(days=lookback_days)).to_period("M")
    end = (test_start - pd.Timedelta(days=1)).to_period("M")
    periods = pd.period_range(start, end, freq="M")
    return [(int(p.year), int(p.month)) for p in periods]

def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def in_range(y, m, sy, sm, ey, em) -> bool:
    return (y > sy or (y == sy and m >= sm)) and (y < ey or (y == ey and m <= em))

# Funtion RandomForest
def train_best_model(train_X, train_y, test_X, test_y, n_grid):
    best_rmse_val = np.inf
    best_mae_val = None
    best_pred = None
    best_n = None

    for n_est in n_grid:
        params = dict(base_rf_params)
        params["n_estimators"] = int(n_est)

        # RandomForestRegressor
        model = RandomForestRegressor(**params)
        model.fit(train_X, train_y)

        pred = model.predict(test_X).astype(np.float32)
        cur_rmse = rmse(test_y, pred)

        if cur_rmse < best_rmse_val:
            best_rmse_val = cur_rmse
            best_mae_val = float(mean_absolute_error(test_y, pred))
            best_pred = pred
            best_n = int(n_est)

    return best_pred, best_mae_val, best_rmse_val, best_n

def build_sample_weights(pdf: pd.DataFrame, weight_event: float, event_col: str):
    w = np.ones(len(pdf), dtype=np.float32)
    if event_col in pdf.columns and weight_event is not None and weight_event > 1:
        # Convertimos a numérico por seguridad y comparamos
        ev = pd.to_numeric(pdf[event_col], errors='coerce').fillna(0).astype(np.int8).values
        w[ev == 1] = float(weight_event)
    return w

# ------------------------------------------------------------
# 3) BUILD FEATURES FROM GOLD_V2 (lags + rolling) -> FEATURES_DIR
# ------------------------------------------------------------
if REBUILD_FEATURES:
    if not path_exists(GOLD_V2_DIR):
        raise Exception(f"GOLD_V2_DIR not found: {GOLD_V2_DIR}")

    print("REBUILD_FEATURES=True -> building features from:", GOLD_V2_DIR)
    
     parent_dir = "/".join(FEATURES_DIR.split("/")[:-1])
    dbutils.fs.mkdirs(parent_dir)

    df = spark.read.parquet(GOLD_V2_DIR)

    # Downtown filter (lat/lon)
    df = df.filter(
        (F.col("lat").between(DOWNTOWN_LAT_MIN, DOWNTOWN_LAT_MAX)) &
        (F.col("lon").between(DOWNTOWN_LON_MIN, DOWNTOWN_LON_MAX))
    )

    print("Stations (downtown):", df.select("station_id").distinct().count())
    print("Rows (downtown):", df.count())

    # Create date and dow_num (Spark dayofweek: Sun=1..Sat=7)
    df = (
        df
        .withColumn("date", F.make_date("year", "month", "day"))
        .withColumn("dow_num", F.dayofweek("date"))
    )

    # Window by station ordered by time
    w = Window.partitionBy("station_id").orderBy(F.col("date"), F.col("hour"))

    # Lags for departures + arrivals
    df = (
        df
        .withColumn("lag1_dep",   F.lag("departures", 1).over(w))
        .withColumn("lag2_dep",   F.lag("departures", 2).over(w))
        .withColumn("lag24_dep",  F.lag("departures", 24).over(w))
        .withColumn("lag168_dep", F.lag("departures", 168).over(w))

        .withColumn("lag1_arr",   F.lag("arrivals", 1).over(w))
        .withColumn("lag2_arr",   F.lag("arrivals", 2).over(w))
        .withColumn("lag24_arr",  F.lag("arrivals", 24).over(w))
        .withColumn("lag168_arr", F.lag("arrivals", 168).over(w))
    )

    # Rolling windows (past only)
    roll_w_3h  = w.rowsBetween(-3,  -1)
    roll_w_24h = w.rowsBetween(-24, -1)

    df = (
        df
        .withColumn("roll_mean_3h_dep",  F.avg("departures").over(roll_w_3h))
        .withColumn("roll_std_24h_dep",  F.stddev("departures").over(roll_w_24h))

        .withColumn("roll_mean_3h_arr",  F.avg("arrivals").over(roll_w_3h))
        .withColumn("roll_std_24h_arr",  F.stddev("arrivals").over(roll_w_24h))
    )
    
    df_feat = df.dropna()

    (df_feat.write
        .mode("overwrite")
        .partitionBy("year", "month")
        .parquet(FEATURES_DIR))

    print("Features successfully written to:", FEATURES_DIR)

# ------------------------------------------------------------
# 4) LOAD FEATURES 
# ------------------------------------------------------------
if not path_exists(FEATURES_DIR):
   
    if not REBUILD_FEATURES:
        raise Exception(
            f"FEATURES_DIR not found: {FEATURES_DIR}\n"
            "Set REBUILD_FEATURES=True to build them."
        )
    else:
        raise Exception(f"Critical Error: Failed to create FEATURES_DIR even with REBUILD_FEATURES=True")

df_feat = spark.read.parquet(FEATURES_DIR)
loaded_rows = df_feat.count()
print("Loaded feature rows:", f"{loaded_rows:,}")

# ------------------------------------------------------------
# 5) Define columns & 6) Month list (Se mantienen igual)
# ------------------------------------------------------------
weather_cols = ["temperature_2m_celsius", "apparent_temperature_celsius"]
event_numeric_cols = [
    "event_day_flag", "event_day_attendance_sum", "events_day_count",
    "event_active_nearby_flag", "events_nearby_count", "nearest_event_km",
    "event_weighted_intensity", "event_attendance_est_sum_nearby", "event_impact_score",
]
base_cols = ["station_id", "year", "month", "day", "hour", "date", "dow_num"]
DEP_TGT, ARR_TGT = "departures", "arrivals"
dep_hist = ["lag1_dep","lag2_dep","lag24_dep","lag168_dep","roll_mean_3h_dep","roll_std_24h_dep"]
arr_hist = ["lag1_arr","lag2_arr","lag24_arr","lag168_arr","roll_mean_3h_arr","roll_std_24h_arr"]

common_feats = [
    BUCKET_COL, "month", "hour", "dow_num", "is_weekend",
    "temperature_2m_celsius", "apparent_temperature_celsius",
    "event_day_flag", "event_day_attendance_sum", "events_day_count",
    "event_active_nearby_flag", "events_nearby_count", "nearest_event_km",
    "event_weighted_intensity", "event_attendance_est_sum_nearby",
    "event_impact_score",
]
dep_model_features = common_feats + dep_hist
arr_model_features = common_feats + arr_hist


# ------------------------------------------------------------
# 6) Month list (full period)
# ------------------------------------------------------------
months_rows = df_feat.select("year", "month").distinct().collect()
months_list = sorted([(int(r["year"]), int(r["month"])) for r in months_rows])
months_list = [(y,m) for (y,m) in months_list if in_range(y,m, START_Y, START_M, END_Y, END_M)]
print("Months to evaluate:", len(months_list), "|", months_list[0], "->", months_list[-1])


# ------------------------------------------------------------
# 7) Pandas month cache (1 toPandas per month)
# ------------------------------------------------------------
month_cache = {}

select_cols_for_pandas = (
    base_cols
    + weather_cols
    + event_numeric_cols
    + dep_hist
    + arr_hist
    + [DEP_TGT, ARR_TGT]
)

def load_month_pd(y:int, m:int) -> pd.DataFrame:
    key = (y,m)
    if key in month_cache:
        return month_cache[key]

    sdf = (df_feat
        .filter((F.col("year")==y) & (F.col("month")==m))
        .select(select_cols_for_pandas)
    )

    pdf = sdf.toPandas()
    if len(pdf) > 0:
        pdf["date"] = pd.to_datetime(pdf["date"])

        # Hash bucket
        pdf[BUCKET_COL] = pdf["station_id"].map(lambda s: station_to_bucket(s, HASH_BUCKETS)).astype(np.int16)

        # Weekend flag (Spark convention: Sun=1, Sat=7)
        pdf["is_weekend"] = pdf["dow_num"].isin([1,7]).astype(np.int8)

        # Fill numeric nulls defensively
        for c in weather_cols + event_numeric_cols + dep_hist + arr_hist:
            if c in pdf.columns:
                pdf[c] = pdf[c].fillna(0)

        # nearest_event_km: sentinel 999.0 already exists for no event,
        # keep as is (it's informative)
    month_cache[key] = pdf
    return pdf

def build_xy(pdf: pd.DataFrame, features: list, target: str):
    missing = [c for c in features + [target] if c not in pdf.columns]
    if missing:
        raise KeyError(f"Missing cols in pandas for target='{target}': {missing}")
    X = pdf[features].astype(np.float32).values
    y = pdf[target].astype(np.float32).values
    return X, y



# ------------------------------------------------------------
# 8) Progressive monthly training/eval (DEP + ARR)
# ------------------------------------------------------------
results_dep = []
results_arr = []

for (y, m) in months_list:
    test_start = pd.Timestamp(year=y, month=m, day=1)
    test_end   = test_start + pd.offsets.MonthEnd(0)

    train_end   = test_start - pd.Timedelta(days=1)
    train_start = train_end - pd.Timedelta(days=TRAIN_LOOKBACK_DAYS)

    test_pd = load_month_pd(y, m)
    if test_pd.empty:
        continue

    # Build train from lookback months
    train_parts = []
    for (yy, mm) in prev_months_in_lookback(test_start, TRAIN_LOOKBACK_DAYS):
        if not in_range(yy, mm, START_Y, START_M, END_Y, END_M):
            continue
        part = load_month_pd(yy, mm)
        if not part.empty:
            train_parts.append(part)

    if not train_parts:
        continue

    train_all = pd.concat(train_parts, ignore_index=True)

    train_pd = train_all[(train_all["date"] >= train_start) & (train_all["date"] <= train_end)]
    test_pd_f = test_pd[(test_pd["date"] >= test_start) & (test_pd["date"] <= test_end)]

    if train_pd.empty or test_pd_f.empty:
        continue

    # Align ordering for hygiene
    join_keys = ["station_id","date","hour"]
    train_pd = train_pd.sort_values(join_keys).reset_index(drop=True)
    test_pd_f = test_pd_f.sort_values(join_keys).reset_index(drop=True)

    # ---------------- DEP ----------------
    dep_train_X, dep_train_y = build_xy(train_pd, dep_model_features, DEP_TGT)
    dep_test_X, dep_test_y   = build_xy(test_pd_f, dep_model_features, DEP_TGT)

    dep_baseline = test_pd_f["lag1_dep"].astype(np.float32).values
    dep_baseline_mae = float(mean_absolute_error(dep_test_y, dep_baseline))
    dep_baseline_rmse = rmse(dep_test_y, dep_baseline)

    dep_pred, dep_mae, dep_rmse, dep_best_n = train_best_model(
        dep_train_X, dep_train_y, dep_test_X, dep_test_y, N_ESTIMATORS_GRID
    )
    dep_impr = (dep_baseline_mae - dep_mae) / dep_baseline_mae * 100 if dep_baseline_mae else np.nan

    results_dep.append({
        "year": y, "month": m,
        "rows_test": int(len(dep_test_y)),
        "baseline_mae": dep_baseline_mae,
        "model_mae": float(dep_mae),
        "baseline_rmse": dep_baseline_rmse,
        "model_rmse": float(dep_rmse),
        "improvement_pct": float(dep_impr),
        "best_n_estimators": int(dep_best_n),
        "hash_buckets": int(HASH_BUCKETS),
        "train_lookback_days": int(TRAIN_LOOKBACK_DAYS)
    })

    # ---------------- ARR ----------------
    arr_train_X, arr_train_y = build_xy(train_pd, arr_model_features, ARR_TGT)
    arr_test_X, arr_test_y   = build_xy(test_pd_f, arr_model_features, ARR_TGT)

    arr_baseline = test_pd_f["lag1_arr"].astype(np.float32).values
    arr_baseline_mae = float(mean_absolute_error(arr_test_y, arr_baseline))
    arr_baseline_rmse = rmse(arr_test_y, arr_baseline)

    arr_pred, arr_mae, arr_rmse, arr_best_n = train_best_model(
        arr_train_X, arr_train_y, arr_test_X, arr_test_y, N_ESTIMATORS_GRID
    )
    arr_impr = (arr_baseline_mae - arr_mae) / arr_baseline_mae * 100 if arr_baseline_mae else np.nan

    results_arr.append({
        "year": y, "month": m,
        "rows_test": int(len(arr_test_y)),
        "baseline_mae": arr_baseline_mae,
        "model_mae": float(arr_mae),
        "baseline_rmse": arr_baseline_rmse,
        "model_rmse": float(arr_rmse),
        "improvement_pct": float(arr_impr),
        "best_n_estimators": int(arr_best_n),
        "hash_buckets": int(HASH_BUCKETS),
        "train_lookback_days": int(TRAIN_LOOKBACK_DAYS)
    })


# ------------------------------------------------------------
# 9) Show + Save
# ------------------------------------------------------------
dep_pd = pd.DataFrame(results_dep).sort_values(["year","month"])
arr_pd = pd.DataFrame(results_arr).sort_values(["year","month"])

print("DEP months evaluated:", len(dep_pd))
print("ARR months evaluated:", len(arr_pd))

display(dep_pd)
display(arr_pd)

spark.createDataFrame(dep_pd).write.mode("overwrite").parquet(EVAL_DIR_DEP)
spark.createDataFrame(arr_pd).write.mode("overwrite").parquet(EVAL_DIR_ARR)

print("Saved DEP eval to:", EVAL_DIR_DEP)
print("Saved ARR eval to:", EVAL_DIR_ARR)

**Net Flow and Serving Preparation**

This section derives net flow outputs from departure and arrival predictions and prepares model artifacts for downstream use.

WEIGHT = 10

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import zlib
import json
import joblib
import os
from datetime import datetime, timezone

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ============================================================
# Net Flow (Derived) - GOLD_V2 (Random Forest Version)
#
# Serverless-safe:
# - 1 toPandas per month (cache in dict)
# - Controlled grid for n_estimators
# - Baseline net = lag1_arr - lag1_dep
# - Weighted training for Event hours via sample_weight
#
# Serving (Serverless-safe):
# - Train FINAL dep/arr models on ALL available data
# - Save models to JOBLIB in DBFS Volumes using local POSIX paths
# - Save feature list metadata (JSON) for safe scoring
# - Validate artifacts after write
# ============================================================

# ------------------------------------------------------------
# 0) Paths
# ------------------------------------------------------------
FEATURES_GOLDV2_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/features/goldv2_features_v1"

USE_BEST_N_FROM_PREV_RUNS = False
EVAL_DEP_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/model_eval/goldv2_dep_rf_v1"
EVAL_ARR_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/model_eval/goldv2_arr_rf_v1"

WEIGHT_EVENT = 10
EVENT_FLAG_COL = "event_active_nearby_flag"

EVAL_DIR_NETFLOW = (
    f"dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/model_eval/"
    f"goldv2_netflow_rf_final_weight{WEIGHT_EVENT}"
)

# Serving model outputs (Unity Catalog Volumes)
SERVING_MODEL_DIR_DBFS = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/models_rf"
DEP_MODEL_BIN_DBFS = f"{SERVING_MODEL_DIR_DBFS}/dep_weight{WEIGHT_EVENT}.joblib"
ARR_MODEL_BIN_DBFS = f"{SERVING_MODEL_DIR_DBFS}/arr_weight{WEIGHT_EVENT}.joblib"

SERVING_META_DIR_DBFS = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/metadata_rf"
FEATURE_META_DIR_DBFS = f"{SERVING_META_DIR_DBFS}/features_weight{WEIGHT_EVENT}"
FEATURE_META_JSON_DBFS = f"{FEATURE_META_DIR_DBFS}/features.json"

MAX_META_BYTES = 500_000 

# ------------------------------------------------------------
# 1) Config
# ------------------------------------------------------------
TRAIN_LOOKBACK_DAYS = 90

HASH_BUCKETS = 512
BUCKET_COL = "station_bucket"

N_ESTIMATORS_GRID = [100, 200, 300]

base_rf_params = dict(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=4,
    max_features="sqrt",
    bootstrap=True,
    n_jobs=-1,
    random_state=42
)

# ------------------------------------------------------------
# 2) Helpers
# ------------------------------------------------------------
def get_local_path(dbfs_path: str) -> str:
    if "dbfs:/Volumes/" in dbfs_path:
        return dbfs_path.replace("dbfs:/Volumes/", "/Volumes/")
    return "/dbfs/" + dbfs_path.replace("dbfs:/", "")

def path_exists(path: str) -> bool:
    try:
        _ = dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def ensure_dir_dbfs(dir_path: str):
    dbutils.fs.mkdirs(dir_path)

def atomic_put_text(dbfs_path: str, text: str):
    dst_dir = dbfs_path.rsplit("/", 1)[0]
    ensure_dir_dbfs(dst_dir)
    tmp = dbfs_path + ".tmp"
    dbutils.fs.put(tmp, text, True)
    try:
        dbutils.fs.rm(dbfs_path, True)
    except Exception:
        pass
    dbutils.fs.mv(tmp, dbfs_path, True)

def read_dbfs_text(dbfs_path: str, max_bytes: int) -> str:
    return dbutils.fs.head(dbfs_path, max_bytes)

def station_to_bucket(station_id: str, n_buckets: int) -> int:
    if station_id is None:
        return 0
    b = str(station_id).encode("utf-8")
    return zlib.crc32(b) % n_buckets

def prev_months_in_lookback(test_start: pd.Timestamp, lookback_days: int):
    start = (test_start - pd.Timedelta(days=lookback_days)).to_period("M")
    end = (test_start - pd.Timedelta(days=1)).to_period("M")
    periods = pd.period_range(start, end, freq="M")
    return [(int(p.year), int(p.month)) for p in periods]

def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def build_xy(pdf: pd.DataFrame, features: list, target: str):
    missing = [c for c in features + [target] if c not in pdf.columns]
    if missing:
        raise KeyError(f"Missing columns for target='{target}': {missing}")
    X = pdf[features].astype(np.float32).values
    y = pdf[target].astype(np.float32).values
    return X, y

def build_sample_weights(pdf: pd.DataFrame, weight_event: float, event_col: str):
    w = np.ones(len(pdf), dtype=np.float32)
    if event_col in pdf.columns and weight_event is not None and weight_event > 1:
        ev = pd.to_numeric(pdf[event_col], errors="coerce").fillna(0).astype(np.int8).values
        w[ev == 1] = float(weight_event)
    return w

def train_best_model(train_X, train_y, test_X, test_y, n_grid, forced_best_n=None, sample_weight=None):
    candidates = [forced_best_n] if (forced_best_n is not None and forced_best_n > 0) else n_grid
    best_rmse_val = np.inf
    best_mae_val, best_pred, best_n = None, None, None

    for n_est in candidates:
        params = dict(base_rf_params)
        params["n_estimators"] = int(n_est)
        model = RandomForestRegressor(**params)
        
        if sample_weight is not None:
            model.fit(train_X, train_y, sample_weight=sample_weight)
        else:
            model.fit(train_X, train_y)

        pred = model.predict(test_X).astype(np.float32)
        cur_rmse = rmse(test_y, pred)

        if cur_rmse < best_rmse_val:
            best_rmse_val = cur_rmse
            best_mae_val = float(mean_absolute_error(test_y, pred))
            best_pred = pred
            best_n = int(n_est)

    return best_pred, best_mae_val, best_rmse_val, best_n

def save_rf_model_joblib_dbfs(model: RandomForestRegressor, dbfs_path: str):
    local_path = get_local_path(dbfs_path)
    local_dir = os.path.dirname(local_path)
    if not os.path.exists(local_dir):
        os.makedirs(local_dir, exist_ok=True)
    joblib.dump(model, local_path)
    print(f"✅ Saved model JOBLIB to: {local_path}")

def save_json_dbfs(obj: dict, dbfs_path: str):
    json_str = json.dumps(obj, indent=2)
    if len(json_str) < 50:
        raise Exception(f"Metadata JSON suspiciously small: {dbfs_path}")
    atomic_put_text(dbfs_path, json_str)
    print(f"✅ Saved metadata JSON to: {dbfs_path} (bytes={len(json_str)})")

def validate_artifacts(dep_path: str, arr_path: str, meta_path: str, expected_n: int):
    for p in [dep_path, arr_path, meta_path]:
        if not path_exists(p):
            raise Exception(f"Validation failed: missing artifact {p}")

    meta = json.loads(read_dbfs_text(meta_path, max_bytes=MAX_META_BYTES))
    dep_feats = meta.get("dep_features", [])
    arr_feats = meta.get("arr_features", [])
    
    def load_local(p): return joblib.load(get_local_path(p))
    dep_m = load_local(dep_path)
    arr_m = load_local(arr_path)

    dep_n = len(dep_m.estimators_)
    arr_n = len(arr_m.estimators_)

    print(f"✅ Validation: meta OK (dep_feats={len(dep_feats)}, arr_feats={len(arr_feats)})")
    print(f"✅ Validation: models OK (dep_n={dep_n}, arr_n={arr_n})")

    if dep_n != expected_n or arr_n != expected_n:
        raise Exception(f"Validation failed: expected {expected_n} estimators")
    print("🎉 ALL ARTIFACT VALIDATIONS PASSED")

# ------------------------------------------------------------
# 3) Load Features & 4) Column Definitions
# ------------------------------------------------------------
if not path_exists(FEATURES_GOLDV2_DIR):
    raise Exception(f"FEATURES_GOLDV2_DIR not found: {FEATURES_GOLDV2_DIR}")

df_feat = spark.read.parquet(FEATURES_GOLDV2_DIR)
dep_target, arr_target = "departures", "arrivals"
key_cols = ["station_id", "year", "month", "day", "hour", "date", "dow_num"]
dep_lag_cols = ["lag1_dep", "lag2_dep", "lag24_dep", "lag168_dep"]
arr_lag_cols = ["lag1_arr", "lag2_arr", "lag24_arr", "lag168_arr"]
dep_roll_cols = ["roll_mean_3h_dep", "roll_std_24h_dep"]
arr_roll_cols = ["roll_mean_3h_arr", "roll_std_24h_arr"]
weather_cols = ["temperature_2m_celsius", "apparent_temperature_celsius"]
event_cols = ["event_day_flag", "events_day_count", "event_day_attendance_sum", EVENT_FLAG_COL, 
              "events_nearby_count", "nearest_event_km", "event_weighted_intensity", 
              "event_attendance_est_sum_nearby", "event_impact_score"]

common_features = [BUCKET_COL, "month", "hour", "dow_num", "is_weekend"] + weather_cols + event_cols
dep_model_features = list(common_features + dep_lag_cols + dep_roll_cols)
arr_model_features = list(common_features + arr_lag_cols + arr_roll_cols)

# ------------------------------------------------------------
# 5) Validate Columns & 6) Month List
# ------------------------------------------------------------
cols_set = set(df_feat.columns)
must_have = key_cols + [dep_target, arr_target] + dep_lag_cols + arr_lag_cols + dep_roll_cols + arr_roll_cols + weather_cols + event_cols
missing = [c for c in must_have if c not in cols_set]
if missing: raise Exception(f"Missing columns: {missing}")

months_list = sorted([(int(r["year"]), int(r["month"])) for r in df_feat.select("year", "month").distinct().collect()])
print(f"Months to evaluate: {len(months_list)}")

# ------------------------------------------------------------
# 8) Pandas Month Cache
# ------------------------------------------------------------
month_cache = {}
numeric_cols = weather_cols + event_cols + dep_lag_cols + arr_lag_cols + dep_roll_cols + arr_roll_cols

def load_month(y: int, m: int) -> pd.DataFrame:
    if (y, m) in month_cache: return month_cache[(y, m)]
    sdf = df_feat.filter((F.col("year") == y) & (F.col("month") == m))
    pdf = sdf.toPandas()
    if len(pdf) > 0:
        pdf["date"] = pd.to_datetime(pdf["date"])
        pdf[BUCKET_COL] = pdf["station_id"].map(lambda s: station_to_bucket(s, HASH_BUCKETS)).astype(np.int16)
        pdf["is_weekend"] = pdf["dow_num"].isin([1, 7]).astype(np.int8)
        for c in numeric_cols: pdf[c] = pd.to_numeric(pdf[c], errors="coerce").fillna(0.0)
    month_cache[(y, m)] = pdf
    return pdf

# ------------------------------------------------------------
# 9) Monthly Loop
# ------------------------------------------------------------
results = []
skipped_no_train = 0

for (y, m) in months_list:
    test_start = pd.Timestamp(year=y, month=m, day=1)
    test_pd = load_month(y, m)
    if test_pd.empty: continue

    train_parts = [load_month(yy, mm) for (yy, mm) in prev_months_in_lookback(test_start, TRAIN_LOOKBACK_DAYS) if not load_month(yy, mm).empty]
    if not train_parts: 
        skipped_no_train += 1
        continue

    train_pd = pd.concat(train_parts, ignore_index=True)
    train_pd = train_pd[(train_pd["date"] < test_start)].sort_values(["station_id", "date", "hour"]).reset_index(drop=True)
    test_pd_f = test_pd.sort_values(["station_id", "date", "hour"]).reset_index(drop=True)

    if train_pd.empty or test_pd_f.empty: 
        skipped_no_train += 1
        continue

    dep_tr_X, dep_tr_y = build_xy(train_pd, dep_model_features, dep_target)
    dep_ts_X, dep_ts_y = build_xy(test_pd_f, dep_model_features, dep_target)
    arr_tr_X, arr_tr_y = build_xy(train_pd, arr_model_features, arr_target)
    arr_ts_X, arr_ts_y = build_xy(test_pd_f, arr_model_features, arr_target)

    train_w = build_sample_weights(train_pd, WEIGHT_EVENT, EVENT_FLAG_COL)

    dep_pred, _, _, d_n = train_best_model(dep_tr_X, dep_tr_y, dep_ts_X, dep_ts_y, N_ESTIMATORS_GRID, sample_weight=train_w)
    arr_pred, _, _, a_n = train_best_model(arr_tr_X, arr_tr_y, arr_ts_X, arr_ts_y, N_ESTIMATORS_GRID, sample_weight=train_w)

    net_real, net_pred = (arr_ts_y - dep_ts_y), (arr_pred - dep_pred)
    baseline_net = (test_pd_f["lag1_arr"] - test_pd_f["lag1_dep"]).astype(np.float32).values

    mae = float(mean_absolute_error(net_real, net_pred))
    baseline_mae = float(mean_absolute_error(net_real, baseline_net))
    
    results.append({
        "year": y, "month": m, "rows_test": len(net_real),
        "baseline_mae_net": baseline_mae, "model_mae_net": mae,
        "dep_best_n": d_n, "arr_best_n": a_n, "weight_event": WEIGHT_EVENT
    })

# ------------------------------------------------------------
# 10) Display Table & Save Parquet
# ------------------------------------------------------------
results_pd = pd.DataFrame(results)
display(results_pd)

results_spark = spark.createDataFrame(results_pd)
results_spark.write.mode("overwrite").parquet(EVAL_DIR_NETFLOW)
print(f"✅ Saved netflow eval parquet to: {EVAL_DIR_NETFLOW}")

# ------------------------------------------------------------
# 11) Train Final Serving Models
# ------------------------------------------------------------
print("\n==================== TRAIN FINAL RF SERVING MODELS ====================")
valid_months = [df for df in month_cache.values() if not df.empty]
full_pd = pd.concat(valid_months, ignore_index=True)
full_w = build_sample_weights(full_pd, WEIGHT_EVENT, EVENT_FLAG_COL)

FINAL_N = 300
dep_final_X, dep_final_y = build_xy(full_pd, dep_model_features, dep_target)
arr_final_X, arr_final_y = build_xy(full_pd, arr_model_features, arr_target)

dep_model = RandomForestRegressor(**{**base_rf_params, "n_estimators": FINAL_N}).fit(dep_final_X, dep_final_y, sample_weight=full_w)
arr_model = RandomForestRegressor(**{**base_rf_params, "n_estimators": FINAL_N}).fit(arr_final_X, arr_final_y, sample_weight=full_w)

save_rf_model_joblib_dbfs(dep_model, DEP_MODEL_BIN_DBFS)
save_rf_model_joblib_dbfs(arr_model, ARR_MODEL_BIN_DBFS)

meta = {
    "created_utc": datetime.now(timezone.utc).isoformat(), "weight_event": WEIGHT_EVENT,
    "hash_buckets": HASH_BUCKETS, "dep_features": dep_model_features, "arr_features": arr_model_features,
    "numeric_cols": numeric_cols
}
save_json_dbfs(meta, FEATURE_META_JSON_DBFS)

validate_artifacts(DEP_MODEL_BIN_DBFS, ARR_MODEL_BIN_DBFS, FEATURE_META_JSON_DBFS, FINAL_N)
print("\n ---- done")

**Artifact Validation**

This section verifies that saved models and feature metadata were correctly written and can be loaded for prediction.

In [0]:
import json
import joblib
import numpy as np
import os

# ============================================================
# Validate Model Artifacts - RANDOM FOREST VERSION
# ============================================================

WEIGHT_EVENT = 10

SERVING_MODEL_DIR_DBFS = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/models_rf"
DEP_MODEL_DBFS = f"{SERVING_MODEL_DIR_DBFS}/dep_weight{WEIGHT_EVENT}.joblib"
ARR_MODEL_DBFS = f"{SERVING_MODEL_DIR_DBFS}/arr_weight{WEIGHT_EVENT}.joblib"

SERVING_META_DIR_DBFS = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/metadata_rf"
FEATURE_META_DIR_DBFS = f"{SERVING_META_DIR_DBFS}/features_weight{WEIGHT_EVENT}"
META_DBFS = f"{FEATURE_META_DIR_DBFS}/features.json"

def get_local_path(dbfs_path: str) -> str:
    """
    Traduce rutas dbfs:/Volumes/... a /Volumes/... para joblib.
    Esto evita el error 'Input/output error' en Unity Catalog.
    """
    if "dbfs:/Volumes/" in dbfs_path:
        return dbfs_path.replace("dbfs:/Volumes/", "/Volumes/")
    return "/dbfs/" + dbfs_path.replace("dbfs:/", "")

def path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def read_dbfs_text(dbfs_path: str, max_bytes: int) -> str:
    return dbutils.fs.head(dbfs_path, max_bytes)

def load_rf_model_from_dbfs(dbfs_path: str):
    """
    Carga el modelo Random Forest usando joblib a través de la ruta local correcta.
    """
    local_path = get_local_path(dbfs_path)
    if not os.path.exists(local_path):
        raise FileNotFoundError(f"No se encontró el archivo en la ruta local: {local_path}")
    return joblib.load(local_path)

print("=== VALIDATE RANDOM FOREST ARTIFACTS ===")

# 1) Paths exist
for p in [DEP_MODEL_DBFS, ARR_MODEL_DBFS, META_DBFS]:
    if not path_exists(p):
        raise Exception(f"Missing artifact: {p}")
print("✅ DBFS paths exist")

# 2) Metadata contract
meta = json.loads(read_dbfs_text(META_DBFS, max_bytes=300_000))
dep_feats = meta["dep_features"]
arr_feats = meta["arr_features"]
hash_buckets = int(meta.get("hash_buckets", 512))

if not isinstance(dep_feats, list) or not isinstance(arr_feats, list):
    raise Exception("dep_features/arr_features must be lists")

print(f"✅ Metadata loaded. dep_features={len(dep_feats)}, arr_features={len(arr_feats)}, hash_buckets={hash_buckets}")

# 3) Load models + sanity checks
dep_model = load_rf_model_from_dbfs(DEP_MODEL_DBFS)
arr_model = load_rf_model_from_dbfs(ARR_MODEL_DBFS)

dep_estimators = len(dep_model.estimators_)
arr_estimators = len(arr_model.estimators_)

print("✅ Random Forest models loaded in memory")
print("dep num_estimators:", dep_estimators)
print("arr num_estimators:", arr_estimators)

# 4) Quick smoke prediction
X_dep = np.zeros((2, len(dep_feats)), dtype=np.float32)
X_arr = np.zeros((2, len(arr_feats)), dtype=np.float32)

dep_pred = dep_model.predict(X_dep)
arr_pred = arr_model.predict(X_arr)

if np.any(np.isnan(dep_pred)) or np.any(np.isnan(arr_pred)):
    raise Exception("NaNs in smoke predictions")

print("✅ Smoke prediction OK (no NaNs)")
print(f"Sample prediction (DEP): {dep_pred[0]:.4f}")
print(f"Sample prediction (ARR): {arr_pred[0]:.4f}")

print("🎉 RF MODEL ARTIFACTS LOOK PERFECT")

**Results and Interpretation**

The Random Forest models capture temporal patterns in both departures and arrivals using non-linear relationships.

By extending beyond single-target modeling, this approach enables the derivation of net flow signals, providing a more comprehensive view of station-level demand dynamics.

This notebook represents a transition from simple predictive models toward more integrated modeling approaches used in later stages of the project.